# Phase 3C - Aggregate & memory-debug

**Phase 3C - Aggregate the sweep + memory-debug loop.** Independent notebook - runs standalone in Colab or locally.

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag" # change in scripts/build_phase_notebooks.py to retarget everywhere
try:
 import google.colab # noqa
 IN_COLAB = True
except Exception:
 IN_COLAB = False
if IN_COLAB:
 repo = Path("/content/kdd26-memdiag")
 if not repo.exists():
 subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
 SOURCE = repo / "experiment" / "github_submission" / "source"
 subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
 SOURCE = None
 for cand in [Path.cwd(), *Path.cwd().parents]:
 for sub in ("source", "experiment"):
 if (cand / sub / "run.py").exists():
 SOURCE = cand / sub
 break
 if SOURCE:
 break
 if SOURCE is None:
 raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)


## Aggregate the shared leaderboard

In [ ]:
import json, pathlib
lb = SOURCE_DIR / 'data' / 'topics' / 'autoresearch' / 'leaderboard'
rows = []
for f in sorted(lb.glob('*.json')):
 d = json.loads(f.read_text())
 for t in d.get('trials', []):
 if t.get('val_bpb') is not None:
 rows.append((t['config']['depth'], t['config']['device_batch_size'], round(t['val_bpb'],4), d.get('group','?')))
rows.sort()
for depth,batch,bpb,g in rows: print(f'depth {depth} batch {batch} val_bpb {bpb} (group {g})')
if rows: print('Best:', min(rows, key=lambda r: r[2]))
else: print('(no group files yet - run the group notebook first, or restore sample leaderboard/A.json)')

## Memory-debug loop (keep / revise / discard) + plot

In [ ]:
if rows:
 bb=min(r[2] for r in rows)
 for d,b,p,g in rows: print(f'depth {d} batch {b} val_bpb {p:.3f} ->', 'keep' if p<=bb*1.05 else ('revise' if p<=bb*1.25 else 'discard'))
try:
 import matplotlib.pyplot as plt
 if rows:
 rs=sorted(rows); plt.figure(figsize=(7,4)); plt.plot([r[0] for r in rs],[r[2] for r in rs],'o-')
 plt.xlabel('depth'); plt.ylabel('val_bpb'); plt.title('Phase 3 sweep'); plt.grid(alpha=0.3); plt.show()
except Exception as e: print('plot skipped:', e)

### Group presentations (3 min each) & wrap-up